In [1]:
import ast
import pandas as pd
import numpy as np

In [2]:
def count_items(value):
    if pd.isna(value):
        return 0

    text = str(value).strip()
    if text in ('', '[]', 'nan', 'None'):
        return 0

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set, dict)):
            return len(parsed)
    except (ValueError, SyntaxError):
        pass

    cleaned = text.strip('[]')
    if not cleaned:
        return 0

    return len([item for item in cleaned.split(',') if item.strip()])

chunk_columns = ['abstract', 'authors', 'n_citation', 'references', 'title', 'venue', 'year', 'id']
chunks = []
for chunk in pd.read_csv('dblp-v10.csv', usecols=chunk_columns, chunksize=50000, engine='python'):
    chunk = chunk.drop_duplicates().copy()
    text_columns = ['abstract', 'authors', 'references', 'title', 'venue']
    chunk[text_columns] = chunk[text_columns].fillna('')
    chunk['year'] = pd.to_numeric(chunk['year'], errors='coerce')
    chunk['n_citation'] = pd.to_numeric(chunk['n_citation'], errors='coerce')
    chunk = chunk.dropna(subset=['year', 'n_citation']).copy()
    chunk['year'] = chunk['year'].astype(int)
    chunk['reference_count'] = chunk['references'].apply(count_items)
    chunk['author_count'] = chunk['authors'].apply(count_items)
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True).drop_duplicates().copy()
df = df[df['venue'].str.strip() != ''].copy()

print('Cleaned rows:', len(df))
print('Duplicate rows:', df.duplicated().sum())
print('Missing values per column:')
print(df.isna().sum())

assert df.isna().sum().sum() == 0
assert df.duplicated().sum() == 0

df.head()

Cleaned rows: 822245
Duplicate rows: 0
Missing values per column:
abstract           0
authors            0
n_citation         0
references         0
title              0
venue              0
year               0
id                 0
reference_count    0
author_count       0
dtype: int64


,abstract,authors,n_citation,references,title,venue,year,id,reference_count,author_count
0,"In this paper, a robust 3D triangular mesh wat...","['S. Ben Jabra', 'Ezzeddine Zagrouba']",50,"['09cb2d7d-47d1-4a85-bfe5-faa8221e644b', '10aa...",A new approach of 3D watermarking based on ima...,international symposium on computers and commu...,2008,4ab3735c-80f1-472d-b953-fa0557fed28b,7,2
1,We studied an autoassociative neural network w...,"['Joaquín J. Torres', 'Jesús M. Cortés', 'Joaq...",50,"['4017c9d2-9845-4ad2-ad5b-ba65523727c5', 'b118...",Attractor neural networks with activity-depend...,Neurocomputing,2007,4ab39729-af77-46f7-a662-16984fb9c1db,3,4
2,It is well-known that Sturmian sequences are t...,"['Genevi eve Paquin', 'Laurent Vuillon']",50,"['1c655ee2-067d-4bc4-b8cc-bc779e9a7f10', '2e4e...",A characterization of balanced episturmian seq...,Electronic Journal of Combinatorics,2007,4ab3a4cf-1d96-4ce5-ab6f-b3e19fc260de,7,2
3,One of the fundamental challenges of recognizi...,"['Yaser Sheikh', 'Mumtaz Sheikh', 'Mubarak Shah']",221,"['056116c1-9e7a-4f9b-a918-44eb199e67d6', '05ac...",Exploring the space of a human action,international conference on computer vision,2005,4ab3a98c-3620-47ec-b578-884ecf4a6206,10,3
4,This paper generalizes previous optimal upper ...,"['Efraim Laksman', 'Håkan Lennerstad', 'Magnus...",0,"['01a765b8-0cb3-495c-996f-29c36756b435', '5dbc...",Generalized upper bounds on the minimum distan...,Ima Journal of Mathematical Control and Inform...,2015,4ab3b585-82b4-4207-91dd-b6bce7e27c4e,9,3


Using pandas to create dummy variables

In [3]:
venue_counts = df['venue'].value_counts()
top_venues = venue_counts.head(10).index
df_top = df[df['venue'].isin(top_venues)].copy()
df_top['high_impact'] = (df_top['n_citation'] > df_top['n_citation'].median()).astype(int)
df_top = df_top[['venue', 'year', 'reference_count', 'author_count', 'high_impact']].copy()
df_top

,venue,year,reference_count,author_count,high_impact
8,international conference on robotics and autom...,1998,4,4,1
42,international conference on communications,2008,8,2,0
51,international conference on communications,2004,12,1,1
57,Bioinformatics,2005,3,3,1
72,international conference on communications,2008,3,5,1
...,...,...,...,...,...
999709,"international conference on acoustics, speech,...",2017,0,3,0
999865,Lecture Notes in Computer Science,2016,0,2,0
999909,Lecture Notes in Computer Science,2016,0,3,0
999923,Lecture Notes in Computer Science,2010,0,4,0


In [4]:
dummies = pd.get_dummies(df_top['venue'])
dummies

,Bioinformatics,Lecture Notes in Computer Science,conference on decision and control,global communications conference,intelligent robots and systems,"international conference on acoustics, speech, and signal processing",international conference on communications,international conference on image processing,international conference on robotics and automation,international geoscience and remote sensing symposium
8,False,False,False,False,False,False,False,False,True,False
42,False,False,False,False,False,False,True,False,False,False
51,False,False,False,False,False,False,True,False,False,False
57,True,False,False,False,False,False,False,False,False,False
72,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...
999709,False,False,False,False,False,True,False,False,False,False
999865,False,True,False,False,False,False,False,False,False,False
999909,False,True,False,False,False,False,False,False,False,False
999923,False,True,False,False,False,False,False,False,False,False


In [5]:
merged = pd.concat([df_top, dummies], axis='columns')
merged

,venue,year,reference_count,author_count,high_impact,Bioinformatics,Lecture Notes in Computer Science,conference on decision and control,global communications conference,intelligent robots and systems,"international conference on acoustics, speech, and signal processing",international conference on communications,international conference on image processing,international conference on robotics and automation,international geoscience and remote sensing symposium
8,international conference on robotics and autom...,1998,4,4,1,False,False,False,False,False,False,False,False,True,False
42,international conference on communications,2008,8,2,0,False,False,False,False,False,False,True,False,False,False
51,international conference on communications,2004,12,1,1,False,False,False,False,False,False,True,False,False,False
57,Bioinformatics,2005,3,3,1,True,False,False,False,False,False,False,False,False,False
72,international conference on communications,2008,3,5,1,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999709,"international conference on acoustics, speech,...",2017,0,3,0,False,False,False,False,False,True,False,False,False,False
999865,Lecture Notes in Computer Science,2016,0,2,0,False,True,False,False,False,False,False,False,False,False
999909,Lecture Notes in Computer Science,2016,0,3,0,False,True,False,False,False,False,False,False,False,False
999923,Lecture Notes in Computer Science,2010,0,4,0,False,True,False,False,False,False,False,False,False,False


In [6]:
final = merged.drop(['venue'], axis='columns')
final

,year,reference_count,author_count,high_impact,Bioinformatics,Lecture Notes in Computer Science,conference on decision and control,global communications conference,intelligent robots and systems,"international conference on acoustics, speech, and signal processing",international conference on communications,international conference on image processing,international conference on robotics and automation,international geoscience and remote sensing symposium
8,1998,4,4,1,False,False,False,False,False,False,False,False,True,False
42,2008,8,2,0,False,False,False,False,False,False,True,False,False,False
51,2004,12,1,1,False,False,False,False,False,False,True,False,False,False
57,2005,3,3,1,True,False,False,False,False,False,False,False,False,False
72,2008,3,5,1,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999709,2017,0,3,0,False,False,False,False,False,True,False,False,False,False
999865,2016,0,2,0,False,True,False,False,False,False,False,False,False,False
999909,2016,0,3,0,False,True,False,False,False,False,False,False,False,False
999923,2010,0,4,0,False,True,False,False,False,False,False,False,False,False


In [7]:
final = final.drop(final.columns[4], axis='columns')
final

,year,reference_count,author_count,high_impact,Lecture Notes in Computer Science,conference on decision and control,global communications conference,intelligent robots and systems,"international conference on acoustics, speech, and signal processing",international conference on communications,international conference on image processing,international conference on robotics and automation,international geoscience and remote sensing symposium
8,1998,4,4,1,False,False,False,False,False,False,False,True,False
42,2008,8,2,0,False,False,False,False,False,True,False,False,False
51,2004,12,1,1,False,False,False,False,False,True,False,False,False
57,2005,3,3,1,False,False,False,False,False,False,False,False,False
72,2008,3,5,1,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
999709,2017,0,3,0,False,False,False,False,True,False,False,False,False
999865,2016,0,2,0,True,False,False,False,False,False,False,False,False
999909,2016,0,3,0,True,False,False,False,False,False,False,False,False
999923,2010,0,4,0,True,False,False,False,False,False,False,False,False


In [8]:
X = final.drop('high_impact', axis='columns')
X

,year,reference_count,author_count,Lecture Notes in Computer Science,conference on decision and control,global communications conference,intelligent robots and systems,"international conference on acoustics, speech, and signal processing",international conference on communications,international conference on image processing,international conference on robotics and automation,international geoscience and remote sensing symposium
8,1998,4,4,False,False,False,False,False,False,False,True,False
42,2008,8,2,False,False,False,False,False,True,False,False,False
51,2004,12,1,False,False,False,False,False,True,False,False,False
57,2005,3,3,False,False,False,False,False,False,False,False,False
72,2008,3,5,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
999709,2017,0,3,False,False,False,False,True,False,False,False,False
999865,2016,0,2,True,False,False,False,False,False,False,False,False
999909,2016,0,3,True,False,False,False,False,False,False,False,False
999923,2010,0,4,True,False,False,False,False,False,False,False,False


In [9]:
y = final.high_impact
y

8         1
42        0
51        1
57        1
72        1
         ..
999709    0
999865    0
999909    0
999923    0
999984    0
Name: high_impact, Length: 77340, dtype: int32

In [10]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=2000)

In [11]:
model.fit(X, y)

LogisticRegression(max_iter=2000)

In [12]:
model.predict(X.head())

array([1, 0, 1, 1, 0])

In [13]:
model.score(X, y)

0.6576545125420222

In [14]:
sample_row = X.head(1).copy()
model.predict(sample_row)

array([1])

Using sklearn OneHotEncoder

In [15]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [16]:
dfle = df_top.copy()
dfle['venue_code'] = le.fit_transform(dfle['venue'])
dfle

,venue,year,reference_count,author_count,high_impact,venue_code
8,international conference on robotics and autom...,1998,4,4,1,8
42,international conference on communications,2008,8,2,0,6
51,international conference on communications,2004,12,1,1,6
57,Bioinformatics,2005,3,3,1,0
72,international conference on communications,2008,3,5,1,6
...,...,...,...,...,...,...
999709,"international conference on acoustics, speech,...",2017,0,3,0,5
999865,Lecture Notes in Computer Science,2016,0,2,0,1
999909,Lecture Notes in Computer Science,2016,0,3,0,1
999923,Lecture Notes in Computer Science,2010,0,4,0,1


In [17]:
X = dfle[['venue_code', 'year', 'reference_count', 'author_count']].values

In [18]:
X

array([[   8, 1998,    4,    4],
       [   6, 2008,    8,    2],
       [   6, 2004,   12,    1],
       ...,
       [   1, 2016,    0,    3],
       [   1, 2010,    0,    4],
       [   8, 2017,   23,    2]], dtype=int64)

In [19]:
y = dfle['high_impact'].values
y

array([1, 0, 1, ..., 0, 0, 0])

Now use one hot encoder to create dummy variables for each venue

In [20]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
ct = ColumnTransformer([('venue', OneHotEncoder(), [0])], remainder='passthrough')

In [21]:
X = ct.fit_transform(X)
X

array([[0.000e+00, 0.000e+00, 0.000e+00, ..., 1.998e+03, 4.000e+00,
        4.000e+00],
       [0.000e+00, 0.000e+00, 0.000e+00, ..., 2.008e+03, 8.000e+00,
        2.000e+00],
       [0.000e+00, 0.000e+00, 0.000e+00, ..., 2.004e+03, 1.200e+01,
        1.000e+00],
       ...,
       [0.000e+00, 1.000e+00, 0.000e+00, ..., 2.016e+03, 0.000e+00,
        3.000e+00],
       [0.000e+00, 1.000e+00, 0.000e+00, ..., 2.010e+03, 0.000e+00,
        4.000e+00],
       [0.000e+00, 0.000e+00, 0.000e+00, ..., 2.017e+03, 2.300e+01,
        2.000e+00]])

In [22]:
X = X[:, 1:]

In [23]:
X

array([[0.000e+00, 0.000e+00, 0.000e+00, ..., 1.998e+03, 4.000e+00,
        4.000e+00],
       [0.000e+00, 0.000e+00, 0.000e+00, ..., 2.008e+03, 8.000e+00,
        2.000e+00],
       [0.000e+00, 0.000e+00, 0.000e+00, ..., 2.004e+03, 1.200e+01,
        1.000e+00],
       ...,
       [1.000e+00, 0.000e+00, 0.000e+00, ..., 2.016e+03, 0.000e+00,
        3.000e+00],
       [1.000e+00, 0.000e+00, 0.000e+00, ..., 2.010e+03, 0.000e+00,
        4.000e+00],
       [0.000e+00, 0.000e+00, 0.000e+00, ..., 2.017e+03, 2.300e+01,
        2.000e+00]])

In [24]:
model.fit(X, y)

c:\Users\UTHANDAM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=2000)

In [25]:
sample_ohe = X[:1]
model.predict(sample_ohe)

array([1])

In [26]:
model.score(X, y)

0.6574993535040082